# Scope definition and data verification

## Imports and parameters

Column names are normalized to uppercase so files from different years can be processed consistently.

In [27]:
from pathlib import Path
import pandas as pd

DATA_FOLDER = Path("dados")
MUNICIPALITIES = ["BRAGA", "GUIMARAES", "GUARDA", "FARO"]

MUNICIPALITIES_AREA_KM2 = {
    "BRAGA": 183,
    "GUIMARAES": 241,
    "GUARDA": 712,
    "FARO": 203,
}



## Reading and combining files

CSV and Excel files are searched for in the `dados` folder and its subfolders. Each file is read separately and the tables are concatenated at the end.

In [28]:

files = sorted(
    file
    for extension in ("csv", "xlsx")
    for file in DATA_FOLDER.glob(f"*.{extension}")
)

if not files:
    raise FileNotFoundError(f"No files found in folder '{DATA_FOLDER}'.")

tables = []

for file in files:
    print("Reading:", file)

    if file.suffix.lower() == ".csv":
        table = pd.read_csv(
            file,
            sep=";",
            decimal=",",
            encoding="utf-8-sig",
            low_memory=False
        )
    else:
        table = pd.read_excel(file)

    table.columns = table.columns.astype(str).str.upper().str.strip()
    tables.append(table)

df = pd.concat(tables, ignore_index=True)

print(f"Total records: {len(df)}")
print("Columns:", sorted(list(df.columns)))


Reading: dados\Registos_Incendios_SGIF_2001_2010.csv
Reading: dados\Registos_Incendios_SGIF_2011_2020.csv
Reading: dados\Registos_Incendios_SGIF_2021_2025.csv
Total records: 497478
Columns: ['ANO', 'AREAAGRIC_HA', 'AREAMATO_HA', 'AREAPOV_HA', 'AREATOTAL_HA', 'BUI', 'CLASSEAREA', 'CODCAUSA', 'CODIGO_ANEPC', 'CODIGO_SGIF', 'CONCELHO', 'DATAHORAALERTA', 'DATAHORA_EXTINCAO', 'DATAHORA_PRIMEIRAINTERVENCAO', 'DC', 'DESCRICAOCAUSA', 'DIA', 'DISTRITO', 'DMC', 'DSR', 'DTCCFR', 'DURACAO_HORAS', 'FFMC', 'FONTEALERTA', 'FREGUESIA', 'FWI', 'GRUPOCAUSA', 'HORA', 'INCSUP24HORAS', 'ISI', 'LATITUDE', 'LOCAL', 'LONGITUDE', 'MES', 'RNAP', 'RNMPF', 'TIPOCAUSA', 'X_ETRS89', 'X_MILITAR', 'Y_ETRS89', 'Y_MILITAR']


## Municipality normalization and scope definition

Normalization removes whitespace, converts text to uppercase, and removes accents. Only the four municipalities in the scope are then kept.

In [29]:
# Standardize municipality names by converting to uppercase, trimming spaces,
# and removing accents/special characters to create a consistent matching key.
df["MUNICIPALITY"] = (
    df["CONCELHO"].astype(str).str.upper().str.strip()
    .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")
)

# Identify the municipality whose normalized versions
# match the reference municipality scope and display each unique matches.
matching_spellings = df.loc[
    df["MUNICIPALITY"].isin(MUNICIPALITIES),
    "CONCELHO"
].unique()
print("Matching spellings:", matching_spellings)


# Filter records belonging to the municipalities scope
# and create an independent copy of the resulting subset.
scoped_data = df[df["MUNICIPALITY"].isin(MUNICIPALITIES)].copy()
print(f"Records in scope: {len(scoped_data)}")


Matching spellings: <StringArray>
['Guimarães', 'Guarda', 'Braga', 'Faro']
Length: 4, dtype: str
Records in scope: 15534


## Creating the analysis year

When `DATAHORAALERTA` exists, the year is extracted from the alert date. Otherwise, the `ANO` column is used.

In [30]:
# Create a unified year variable from the alert date when available;
# otherwise, use the existing year column.
if "DATAHORAALERTA" in scoped_data.columns:
    scoped_data["ANALYSIS_YEAR"] = pd.to_datetime(
        scoped_data["DATAHORAALERTA"], errors="coerce", dayfirst=True
    ).dt.year
else:
    scoped_data["ANALYSIS_YEAR"] = scoped_data["ANO"]


# Preview the municipality and analysis year fields.
scoped_data[
    [
        "MUNICIPALITY",
        "ANALYSIS_YEAR"
    ]
].head() #.head() #.tail()

,MUNICIPALITY,ANALYSIS_YEAR
104,GUIMARAES,2001
117,GUIMARAES,2001
127,GUIMARAES,2001
128,GUARDA,2001
165,GUIMARAES,2001


## Cause normalization

Empty values, `nan`, and `None` are treated as missing information. `MISSING_CAUSE` enables those records to be counted and `NORMALIZED_CAUSE` creates an explicit category.

In [31]:
# Clean and standardize cause descriptions, flag missing values,
# and assign a default label to records without cause information.
cause = (
    scoped_data["TIPOCAUSA"].astype(str).str.strip()
    .replace({"": None, "nan": None, "None": None})
)
scoped_data["CAUSE"] = cause.fillna("DESCONHECIDA").str.upper()
scoped_data["MISSING_CAUSE"] = scoped_data["CAUSE"].eq("DESCONHECIDA")

# Preview the original, normalized, and missing-cause indicators.
scoped_data[
    [
        "TIPOCAUSA",
        "CAUSE",
        "MISSING_CAUSE"
    ]
].head()

,TIPOCAUSA,CAUSE,MISSING_CAUSE
104,NaN,DESCONHECIDA,True
117,NaN,DESCONHECIDA,True
127,NaN,DESCONHECIDA,True
128,NaN,DESCONHECIDA,True
165,NaN,DESCONHECIDA,True


## Derived variables for analysis

The time between the alert and the first intervention is calculated, and years are grouped into the periods used by the test.

In [32]:
alert_time = pd.to_datetime(scoped_data["DATAHORAALERTA"], errors="coerce", dayfirst=True)
intervention_time = pd.to_datetime(
    scoped_data["DATAHORA_PRIMEIRAINTERVENCAO"], errors="coerce", dayfirst=True
)
scoped_data["INTERVENTION_TIME_MINUTES"] = (intervention_time - alert_time).dt.total_seconds() / 60

scoped_data["PERIOD"] = pd.cut(
    scoped_data["ANALYSIS_YEAR"], bins=[2000, 2016, 2017, 2025],
    labels=["2001-2016", "2017", "2018-2025"]
)

scoped_data[["ANALYSIS_YEAR", "INTERVENTION_TIME_MINUTES", "PERIOD"]].head()

,ANALYSIS_YEAR,INTERVENTION_TIME_MINUTES,PERIOD
104,2001,NaN,2001-2016
117,2001,NaN,2001-2016
127,2001,NaN,2001-2016
128,2001,NaN,2001-2016
165,2001,NaN,2001-2016


In [33]:
# Calculate the ignition density for a given scope of data.
# This function takes a DataFrame of ignition data and calculates
# the density of ignitions per square kilometer per year for each municipality.
def calculate_ignition_density(
    data,
    start_year=None,
    end_year=None,
    municipalities=MUNICIPALITIES,
    area_by_municipality=MUNICIPALITIES_AREA_KM2,
    year_column="ANALYSIS_YEAR",
):
    scoped = data.copy()

    if start_year is not None:
        scoped = scoped[scoped[year_column] >= start_year]

    if end_year is not None:
        scoped = scoped[scoped[year_column] <= end_year]

    years_in_scope = scoped[year_column].dropna().nunique()

    if years_in_scope == 0:
        raise ValueError("The selected scope contains no valid years.")

    result = (
        scoped.groupby("MUNICIPALITY")
        .size()
        .rename("IGNITIONS")
        .reindex(municipalities, fill_value=0)
        .to_frame()
    )

    result["AREA_KM2"] = result.index.map(area_by_municipality)
    result["YEARS"] = years_in_scope
    result["IGNITION_DENSITY_IGNITIONS_KM2_YEAR"] = (
        result["IGNITIONS"]
        / result["AREA_KM2"]
        / result["YEARS"]
    )

    return result.reset_index()

In [34]:
# All years
ignition_density = calculate_ignition_density(scoped_data)
display(ignition_density)

,MUNICIPALITY,IGNITIONS,AREA_KM2,YEARS,IGNITION_DENSITY_IGNITIONS_KM2_YEAR
0,BRAGA,4279,183,25,0.935301
1,GUIMARAES,8641,241,25,1.434191
2,GUARDA,1806,712,25,0.101461
3,FARO,808,203,25,0.159212


In [35]:
required_columns = [
    "MUNICIPALITY",             # normalized municipality name
    "PERIOD",                   # period category based on analysis year
    "ANALYSIS_YEAR",            # analysis year
    "DATAHORAALERTA",           # alert date and time 
    "CAUSE",                    # normalized cause description
    "MISSING_CAUSE",            # indicator for missing cause information
    "AREATOTAL_HA",             # total burned area in hectares (if available)
    "INTERVENTION_TIME_MINUTES",# intervention time in minutes (if available)
]

missing_columns = [
    column for column in required_columns
    if column not in scoped_data.columns
]

if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

scoped_data = scoped_data[required_columns].copy()

print(f"Filtered data shape: {scoped_data.shape}")
display(scoped_data.head())

Filtered data shape: (15534, 8)


,MUNICIPALITY,PERIOD,ANALYSIS_YEAR,DATAHORAALERTA,CAUSE,MISSING_CAUSE,AREATOTAL_HA,INTERVENTION_TIME_MINUTES
104,GUIMARAES,2001-2016,2001,18/02/2001 22:31,DESCONHECIDA,True,0.10,NaN
117,GUIMARAES,2001-2016,2001,19/02/2001 14:48,DESCONHECIDA,True,0.20,NaN
127,GUIMARAES,2001-2016,2001,19/02/2001 17:53,DESCONHECIDA,True,3.00,NaN
128,GUARDA,2001-2016,2001,19/02/2001 17:55,DESCONHECIDA,True,1.00,NaN
165,GUIMARAES,2001-2016,2001,20/02/2001 17:33,DESCONHECIDA,True,0.15,NaN


In [36]:
# Uncomment the following lines to save the scoped data to a CSV file.
output_file =  "scoped_data_required_columns.csv"

scoped_data.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved {len(scoped_data):,} records to: {output_file}")

Saved 15,534 records to: scoped_data_required_columns.csv


## Verification tables

These tables reproduce the script checks: records, causes, percentage without a cause, missing values, intervention times, and the final test table.

In [37]:
print("1) Records by year and municipality")
display(pd.crosstab(scoped_data["ANALYSIS_YEAR"], scoped_data["MUNICIPALITY"]))

1) Records by year and municipality


MUNICIPALITY,BRAGA,FARO,GUARDA,GUIMARAES
ANALYSIS_YEAR,,,,
2001,285,16,159,833
2002,279,15,154,795
2003,381,14,122,950
2004,253,27,74,593
2005,406,29,137,876
2006,330,69,102,434
2007,216,80,84,383
2008,83,42,142,170
2009,242,61,109,487


In [38]:
print("2) Cause type by municipality")
display(pd.crosstab(scoped_data["CAUSE"], scoped_data["MUNICIPALITY"]))


2) Cause type by municipality


MUNICIPALITY,BRAGA,FARO,GUARDA,GUIMARAES
CAUSE,,,,
DESCONHECIDA,3687,513,823,7910
INTENCIONAL,146,42,230,202
NATURAL,0,0,50,0
NEGLIGENTE,168,248,635,369
REACENDIMENTO,278,5,68,160


In [39]:
print("3) % of records without a cause, by year and municipality")
display((pd.crosstab(
    scoped_data["ANALYSIS_YEAR"], scoped_data["MUNICIPALITY"], values=scoped_data["MISSING_CAUSE"], aggfunc="mean"
) * 100).round(1))


3) % of records without a cause, by year and municipality


MUNICIPALITY,BRAGA,FARO,GUARDA,GUIMARAES
ANALYSIS_YEAR,,,,
2001,98.9,100.0,100.0,99.9
2002,98.9,73.3,90.3,99.9
2003,99.2,100.0,89.3,99.8
2004,99.6,88.9,89.2,99.5
2005,97.5,100.0,94.9,99.2
2006,99.7,98.6,83.3,99.8
2007,90.7,96.2,48.8,98.4
2008,98.8,90.5,25.4,96.5
2009,87.6,95.1,12.8,98.8


In [40]:
columns = [
    "AREATOTAL_HA", "DATAHORAALERTA",
    "DATAHORA_PRIMEIRAINTERVENCAO", "DATAHORA_EXTINCAO", "DURACAO_HORAS",
]
columns = [column for column in columns if column in scoped_data.columns]

print("4) % missing by year")
display((scoped_data.groupby("ANALYSIS_YEAR")[columns].apply(lambda x: x.isna().mean()) * 100).round(1))


4) % missing by year


,AREATOTAL_HA,DATAHORAALERTA
ANALYSIS_YEAR,,
2001,0.0,0.0
2002,0.0,0.0
2003,0.0,0.0
2004,0.0,0.0
2005,0.0,0.0
2006,0.0,0.0
2007,0.0,0.0
2008,0.0,0.0
2009,0.0,0.0


In [41]:
print("5) Time to first intervention (minutes)")
display(scoped_data.groupby("MUNICIPALITY")["INTERVENTION_TIME_MINUTES"].describe().round(1))
print("Negative:", int((scoped_data["INTERVENTION_TIME_MINUTES"] < 0).sum()),
      "| Above 24h:", int((scoped_data["INTERVENTION_TIME_MINUTES"] > 1440).sum()))


5) Time to first intervention (minutes)


,count,mean,std,min,25%,50%,75%,max
MUNICIPALITY,,,,,,,,
BRAGA,2068.0,12.1,7.8,0.0,8.0,11.0,15.0,139.0
FARO,550.0,10.4,5.0,1.0,7.0,10.0,12.0,42.0
GUARDA,915.0,16.2,15.8,0.0,9.0,13.0,20.0,265.0
GUIMARAES,3671.0,12.1,13.4,0.0,7.0,10.0,15.0,615.0


Negative: 0 | Above 24h: 0


In [42]:

hypothesis_data = scoped_data[
    scoped_data["CAUSE"].str.lower().isin(["negligente", "intencional"])
    & (scoped_data["PERIOD"] != "2017")
]
print("6) Negligent vs intentional by period (2017 excluded)")
display(pd.crosstab([hypothesis_data["MUNICIPALITY"], hypothesis_data["PERIOD"]], hypothesis_data["CAUSE"]))


6) Negligent vs intentional by period (2017 excluded)


CAUSE                   INTENCIONAL  NEGLIGENTE
MUNICIPALITY PERIOD                            
BRAGA        2001-2016           60         113
             2018-2025           85          48
FARO         2001-2016            9         113
             2018-2025           33         117
GUARDA       2001-2016          137         454
             2018-2025           66         147
GUIMARAES    2001-2016           35         145
             2018-2025          167         221